In [13]:
import os
import time
import requests  
import pandas as pd
from bs4 import BeautifulSoup
from glob import glob

# ----------------------------------------------------
# 1. 함수들을 코드 상단에 한 번만 정의합니다.
# ----------------------------------------------------
# opengDateBegin=20140903&opengDateEnd=20141013&bidResultCode=2
# 데이터를 요청하는 함수 (날짜를 파라미터로 받도록 수정)
def request_estate_data(start_date, end_date, num_of_rows='15000'):
    serviceKey = '8bea1c13542aaa1626e136b275cde504cb37de3c3a31d19cd15d7759b330efdf'
    base_url = 'http://openapi.d2b.go.kr/openapi/service/BidResultInfoService'
    service_name = 'getDmstcCmpetBidResultList'
    
    params = {
        'serviceKey': serviceKey,
        'opengDateBegin': start_date,
        'opengDateEnd': end_date,
        'numOfRows': num_of_rows,
        'pageNo': '1',
        'bidResultCode' : '2'
    }
    
    try:
        response = requests.get(base_url + service_name, params=params, timeout=30)
        response.raise_for_status()  # HTTP 오류가 발생하면 예외를 발생시킴
        soup = BeautifulSoup(response.text, 'xml')
        return soup
    except requests.exceptions.RequestException as e:
        print(f"{start_date[:4]}년 데이터 요청 중 오류 발생: {e}")
        return None

# XML 아이템 하나를 처리하는 함수
def to_extract_row(item, tag_names, col_names):
    data = {}
    for tag, col in zip(tag_names, col_names):
        tag_obj = item.find(tag)
        data[col] = tag_obj.text.strip() if tag_obj else None
    return data

# Soup 객체를 데이터프레임으로 변환하는 함수
def extract_data_and_DF(soup, tag_names, col_names):
    if not soup or not soup.find('item'):
        return pd.DataFrame() # 데이터가 없으면 빈 데이터프레임 반환
    
    item_list = [to_extract_row(item_tag, tag_names, col_names) for item_tag in soup.find_all('item')]
    return pd.DataFrame(item_list)

# ----------------------------------------------------
# 2. 반복문으로 연도별 데이터 처리 및 저장을 실행합니다.
# ----------------------------------------------------

# 공통으로 사용할 변수들
tag_names = ['pblancSe','pblancNo','pblancOdr','dcsNo','iemNo','bidNm','demandYear','orntCode','ornt','cntrctMth','bidMth','sucbidrDecsnMth','opengDt','opengDate','bidResult']
col_names = ['공고구분','공고번호','공고차수','판단번호','항목번호','입찰명','요구년도','발주기관코드','발주기관','계약방법','입찰방법','낙찰자결정방법','개찰일시','개찰일자','입찰결과']
save_path = "workbookdata"

# 저장 폴더가 없으면 생성
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 2016년부터 2024년까지 반복
for year in range(2016, 2025):
    start_date = f'{year}0101'
    end_date = f'{year}1231'
    
    print(f"--- {year}년 데이터 처리 시작 ---")
    
    # 1. 데이터 요청
    soup = request_estate_data(start_date, end_date)
    
    # 2. 데이터프레임으로 변환
    if soup:
        df_year = extract_data_and_DF(soup, tag_names, col_names)
        
        # 3. CSV 파일로 즉시 저장
        if not df_year.empty:
            file_name = f'{year}년_공고결과_낙찰방법.csv'
            full_path = os.path.join(save_path, file_name)
            df_year.to_csv(full_path, index=False, encoding='cp949')
            print(f"✅ {file_name} 저장 완료! (총 {len(df_year)}개 행)")
        else:
            print(f"⚠️ {year}년 데이터가 비어있어 파일을 저장하지 않습니다.")
    
    # API 서버에 부담을 주지 않기 위해 요청 사이에 잠시 대기
    time.sleep(1) 

print("\n🎉 모든 연도별 데이터 처리가 완료되었습니다.")

--- 2016년 데이터 처리 시작 ---
✅ 2016년_공고결과_낙찰방법.csv 저장 완료! (총 8421개 행)
--- 2017년 데이터 처리 시작 ---
✅ 2017년_공고결과_낙찰방법.csv 저장 완료! (총 9021개 행)
--- 2018년 데이터 처리 시작 ---
✅ 2018년_공고결과_낙찰방법.csv 저장 완료! (총 8966개 행)
--- 2019년 데이터 처리 시작 ---
✅ 2019년_공고결과_낙찰방법.csv 저장 완료! (총 9168개 행)
--- 2020년 데이터 처리 시작 ---
✅ 2020년_공고결과_낙찰방법.csv 저장 완료! (총 8310개 행)
--- 2021년 데이터 처리 시작 ---
✅ 2021년_공고결과_낙찰방법.csv 저장 완료! (총 8131개 행)
--- 2022년 데이터 처리 시작 ---
✅ 2022년_공고결과_낙찰방법.csv 저장 완료! (총 8547개 행)
--- 2023년 데이터 처리 시작 ---
✅ 2023년_공고결과_낙찰방법.csv 저장 완료! (총 7818개 행)
--- 2024년 데이터 처리 시작 ---
✅ 2024년_공고결과_낙찰방법.csv 저장 완료! (총 7117개 행)

🎉 모든 연도별 데이터 처리가 완료되었습니다.


### 하나의 csv로 합치기

In [14]:
path = "workbookdata"
fname = '*년_공고결과_낙찰방법.csv'
full_pattern = os.path.join(path, fname)
fileList = glob(full_pattern)
fileList

['workbookdata\\2016년_공고결과_낙찰방법.csv',
 'workbookdata\\2017년_공고결과_낙찰방법.csv',
 'workbookdata\\2018년_공고결과_낙찰방법.csv',
 'workbookdata\\2019년_공고결과_낙찰방법.csv',
 'workbookdata\\2020년_공고결과_낙찰방법.csv',
 'workbookdata\\2021년_공고결과_낙찰방법.csv',
 'workbookdata\\2022년_공고결과_낙찰방법.csv',
 'workbookdata\\2023년_공고결과_낙찰방법.csv',
 'workbookdata\\2024년_공고결과_낙찰방법.csv',
 'workbookdata\\2025년_공고결과_낙찰방법.csv']

In [15]:
fileList = fileList[0:9]

In [16]:
import pandas as pd
from glob import glob
import os

all_data_frames = []

for file_path in fileList:
    df = pd.read_csv(file_path, header=0, encoding='cp949')
    all_data_frames.append(df)

PBL_df = pd.concat(all_data_frames, ignore_index=True)
PBL_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75499 entries, 0 to 75498
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   공고구분     75499 non-null  object
 1   공고번호     75499 non-null  object
 2   공고차수     75499 non-null  int64 
 3   판단번호     75499 non-null  object
 4   항목번호     75499 non-null  object
 5   입찰명      75499 non-null  object
 6   요구년도     75499 non-null  int64 
 7   발주기관코드   75499 non-null  object
 8   발주기관     75499 non-null  object
 9   계약방법     75499 non-null  object
 10  입찰방법     75499 non-null  object
 11  낙찰자결정방법  75499 non-null  object
 12  개찰일시     75499 non-null  int64 
 13  개찰일자     75499 non-null  int64 
 14  입찰결과     75499 non-null  object
dtypes: int64(4), object(11)
memory usage: 8.6+ MB


In [17]:
PBL_df.head(1)

,공고구분,공고번호,공고차수,판단번호,항목번호,입찰명,요구년도,발주기관코드,발주기관,계약방법,입찰방법,낙찰자결정방법,개찰일시,개찰일자,입찰결과
0,정상공고,PIP0001,1,9100,***,중요시설경계시스템 1차사업,2016,EGH,지휘정찰계약팀 (구:19.09.16),협상에의한계약(서류),총액제,최저가격제,201612301200,20161230,낙찰


In [21]:
# PBL_df = PBL_df[PBL_df['업무구분'] != '용역']
PBL_df['개찰일시'] = pd.to_datetime(PBL_df['개찰일시'].astype(str), format='%Y%m%d%H%M', errors='coerce')
PBL_df['개찰일자'] = pd.to_datetime(PBL_df['개찰일자'].astype(str), format='%Y%m%d', errors='coerce')

In [22]:
PBL_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75499 entries, 0 to 75498
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   공고구분     75499 non-null  object        
 1   공고번호     75499 non-null  object        
 2   공고차수     75499 non-null  int64         
 3   판단번호     75499 non-null  object        
 4   항목번호     75499 non-null  object        
 5   입찰명      75499 non-null  object        
 6   요구년도     75499 non-null  int64         
 7   발주기관코드   75499 non-null  object        
 8   발주기관     75499 non-null  object        
 9   계약방법     75499 non-null  object        
 10  입찰방법     75499 non-null  object        
 11  낙찰자결정방법  75499 non-null  object        
 12  개찰일시     75498 non-null  datetime64[ns]
 13  개찰일자     75499 non-null  datetime64[ns]
 14  입찰결과     75499 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(11)
memory usage: 8.6+ MB


In [23]:
# save_path = "C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data"

PBL_df.to_csv(os.path.join(path, 'Total_공고결과_낙찰방법.csv'), index=False, encoding='cp949')